In [38]:
### Imports

from pathlib import Path
import gradio as gr
from openai import OpenAI
from IPython.display import Markdown
from typing import Any

In [39]:
### LLM setup

OLLAMA = {
    "BASE_URL": "http://localhost:11434/v1",
    "MODEL": "gemma4",
    "API_KEY": "ollama",
}

url, model, apikey = OLLAMA.get("BASE_URL"), OLLAMA.get("MODEL"), OLLAMA.get("API_KEY")

ollama = OpenAI(base_url=url, api_key=apikey)

In [40]:
### Read all employee data into a dictionary

KNOWLEDGE_BASE = Path("../knowledge-base")

knowledge: dict[str, str] = {}

employees = (KNOWLEDGE_BASE / "employees").glob("*")


for filename in employees:
    name = Path(filename).stem.split(' ')[-1]
    with open (filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [ ]:
knowledge

In [51]:
display(Markdown(knowledge.get("lancaster")))

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Analyst at Edge Analytics  
  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.

## Annual Performance History
- **2015**: **Exceeds Expectations**  
  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial funding.  

- **2016**: **Meets Expectations**  
  Growth continued, though challenges arose in operational efficiency that required Avery's attention.  

- **2017**: **Developing**  
  Market competition intensified, and monthly sales metrics were below targets. Avery implemented new strategies which required a steep learning curve.  

- **2018**: **Exceeds Expectations**  
  Under Avery’s pivoted vision, Insurellm launched two new successful products that significantly increased market share.  

- **2019**: **Meets Expectations**  
  Steady growth, however, some team tensions led to a minor drop in employee morale. Avery recognized the need to enhance company culture.  

- **2020**: **Below Expectations**  
  The COVID-19 pandemic posed unforeseen operational difficulties. Avery faced criticism for delayed strategy shifts, although efforts were eventually made to stabilize the company.  

- **2021**: **Exceptional**  
  Avery's decisive transition to remote work and rapid adoption of digital tools led to record-high customer satisfaction levels and increased sales.  

- **2022**: **Satisfactory**  
  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement despite a saturated market.  

- **2023**: **Exceeds Expectations**  
  Market leadership was regained with innovative approaches to personalized insurance solutions. Avery is now recognized in industry publications as a leading voice in Insurance Tech innovation.

## Compensation History
- **2015**: $150,000 base salary + Significant equity stake  
- **2016**: $160,000 base salary + Equity increase  
- **2017**: $150,000 base salary + Decrease in bonus due to performance  
- **2018**: $180,000 base salary + performance bonus of $30,000  
- **2019**: $185,000 base salary + market adjustment + $5,000 bonus  
- **2020**: $170,000 base salary (temporary reduction due to COVID-19)  
- **2021**: $200,000 base salary + performance bonus of $50,000  
- **2022**: $210,000 base salary + retention bonus  
- **2023**: $225,000 base salary + $75,000 performance bonus  

## Other HR Notes
- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  
- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  
- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.
- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  

Avery Lancaster has demonstrated resilience and adaptability throughout her career at Insurellm, positioning the company as a key player in the insurance technology landscape.

In [43]:
### Same thing but for products

products = (KNOWLEDGE_BASE / "products").glob("*")

for filename in products:
    name = Path(filename).stem.split(' ')[-1]
    with open (filename, "r", encoding="utf-8") as f:
        knowledge.setdefault(name.lower(), f.read())

In [44]:
knowledge.keys()

dict_keys(['chen', 'harper', 'thomson', 'foster', 'lancaster', 'walker', 'rodriguez', 'park', 'kim', 'carter', 'tran', 'wilson', 'adams', 'liu', 'blake', 'bishop', 'zhang', 'anderson', 'johnson', 'thompson', "o'brien", 'rivera', 'patel', 'spencer', 'sharma', 'martinez', 'greene', 'trenton', 'williams', 'brooks', 'bizllm', 'carllm', 'claimllm', 'healthllm', 'homellm', 'lifellm', 'markellm', 'rellm'])

In [45]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [ ]:
### Internal helper to fetch the relevant context

def get_relevant_context(message: Any) -> list[str]:
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge.get(word) for word in words if word in knowledge]

In [ ]:
get_relevant_context("Who is Lancaster and what is carllm?")

In [ ]:
### Helper to use context to enrich prompt

def get_additional_context(message: Any) -> str:
    res = get_relevant_context(message)
    if not res:
        return "No additional context relevant to the user's question.."
    return f"The following additional context might be relevant in answering the user's question:\n\n" + "\n\n".join(res)

In [ ]:
### Chat function

def chat(message: Any, history: list[dict]) -> str:
    sys_msg = SYSTEM_PREFIX + get_additional_context(message)
    messages = [{"role": "system", "content": sys_msg}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=model, messages=messages)
    
    return response.choices[0].message.content

In [50]:
### Gradio UI

gr.ChatInterface(fn=chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
